In [1]:
import torch

In [3]:
batch_size = 4
a = torch.randint(0, 10, (batch_size,))
b = torch.randint(0, 10, (batch_size,))

In [4]:
a,b

(tensor([6, 6, 9, 2]), tensor([0, 9, 8, 8]))

In [5]:
true_answer = a + b

In [6]:
true_answer

tensor([ 6, 15, 17, 10])

In [9]:
group_size = 8

a_rep = a.repeat_interleave(group_size)
b_rep = b.repeat_interleave(group_size)

In [11]:
a_rep.shape

torch.Size([32])

In [12]:
true_rep = true_answer.repeat_interleave(group_size)
true_rep

tensor([ 6,  6,  6,  6,  6,  6,  6,  6, 15, 15, 15, 15, 15, 15, 15, 15, 17, 17,
        17, 17, 17, 17, 17, 17, 10, 10, 10, 10, 10, 10, 10, 10])

In [14]:
logits = torch.randn((batch_size * group_size, 19))

In [16]:
logits.shape

torch.Size([32, 19])

In [17]:
from torch.distributions import Categorical

dist = Categorical(logits = logits)

In [25]:
dist

Categorical(probs: torch.Size([32, 19]), logits: torch.Size([32, 19]))

In [20]:
action = dist.sample()

In [21]:
action

tensor([ 7,  9,  3,  8,  7, 17,  8, 15, 11, 15,  6, 14,  7, 13, 12,  4,  9, 16,
        13,  9,  2, 16, 15,  8,  0,  7,  4, 14,  2, 12,  7,  9])

In [27]:
action.shape

torch.Size([32])

In [23]:
log_probs = dist.log_prob(action)

In [24]:
log_probs

tensor([-3.0161, -2.1437, -1.4657, -2.0802, -2.7246, -3.8421, -2.3168, -4.1103,
        -2.3792, -3.7886, -1.9081, -4.6253, -2.5253, -3.5391, -3.6912, -2.4649,
        -1.8207, -3.7293, -3.4245, -2.9409, -0.7719, -3.9779, -1.3069, -3.3845,
        -2.5977, -2.8731, -1.7708, -1.5833, -1.5336, -3.2046, -1.1638, -2.9573])

In [26]:
log_probs.shape

torch.Size([32])

In [29]:
reward = (action == true_rep).float()

In [30]:
reward

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [31]:
reward_grouped = reward.view(batch_size, group_size)

In [32]:
reward_grouped

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [36]:
mean = reward_grouped.mean(dim=1, keepdim=True)

In [38]:
mean

tensor([[0.0000],
        [0.1250],
        [0.0000],
        [0.0000]])

In [37]:
std = reward_grouped.std(dim=1, keepdim=True) + 1e-4

In [39]:
std

tensor([[1.0000e-04],
        [3.5365e-01],
        [1.0000e-04],
        [1.0000e-04]])

In [40]:
advantage = ((reward_grouped - mean) / std).view(-1)

In [41]:
advantage

tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
        -0.3535,  2.4742, -0.3535, -0.3535, -0.3535, -0.3535, -0.3535, -0.3535,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000])